# Working With Text

In [2]:
import os                                                  #library working with files
import urllib.request                                      # library working with downloading files from internet

if not os.path.exists("the-verdict.txt"):

    url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/refs/heads/main/ch02/01_main-chapter-code/the-verdict.txt"

    file_path = "the-verdict.txt"

    urllib.request.urlretrieve(url, file_path)

    print("File downloaded successfully!")
else:
    print("File already exists!")

File already exists!


In [3]:
with open("the-verdict.txt","r",encoding="utf-8") as f:
    raw_text = f.read()

In [4]:
len(raw_text)

20479

# Example of Tokenization

In [6]:
import re                 #py regular expression library
text = "Hello World. This, is a test."
res = re.split(r'(\s)',text)          # regular expression that splits on white spaces
print(res)

['Hello', ' ', 'World.', ' ', 'This,', ' ', 'is', ' ', 'a', ' ', 'test.']


In [7]:
res = re.split(r'([,.]|\s)',text)          # regular expression that splits on white spaces and panchuation 

In [8]:
print(res)

['Hello', ' ', 'World', '.', '', ' ', 'This', ',', '', ' ', 'is', ' ', 'a', ' ', 'test', '.', '']


In [9]:
res = [i for i in res if i.strip()]       #removes white paces
print(res)

['Hello', 'World', '.', 'This', ',', 'is', 'a', 'test', '.']


In [10]:
txt = "Hello World. Is this-- a test?" 
res = re.split(r'([,.:;?_!"()\')]|--|\s)',txt)  # 
res = [i.strip() for i in res if i.strip()]
print(res)

['Hello', 'World', '.', 'Is', 'this', '--', 'a', 'test', '?']


# Continue The main brach 

In [12]:
res = re.split(r'([,.:;?_!"()\')]|--|\s)',raw_text)
res = [i.strip() for i in res if i.strip()]
preProcessed = res  #preprocessed contains all the tokenized text form raw_text

In [13]:
len(preProcessed)

4690

# Converting tokens into token IDs

In [15]:
all_word = sorted(set(preProcessed))
vocab_size = len(all_word)
print(vocab_size)

1130


In [16]:
vocab = {}  # Creating vocabulary to map the test to integer
for integer, token in enumerate(all_word):
    vocab[token] = integer
print(vocab)

{'!': 0, '"': 1, "'": 2, '(': 3, ')': 4, ',': 5, '--': 6, '.': 7, ':': 8, ';': 9, '?': 10, 'A': 11, 'Ah': 12, 'Among': 13, 'And': 14, 'Are': 15, 'Arrt': 16, 'As': 17, 'At': 18, 'Be': 19, 'Begin': 20, 'Burlington': 21, 'But': 22, 'By': 23, 'Carlo': 24, 'Chicago': 25, 'Claude': 26, 'Come': 27, 'Croft': 28, 'Destroyed': 29, 'Devonshire': 30, 'Don': 31, 'Dubarry': 32, 'Emperors': 33, 'Florence': 34, 'For': 35, 'Gallery': 36, 'Gideon': 37, 'Gisburn': 38, 'Gisburns': 39, 'Grafton': 40, 'Greek': 41, 'Grindle': 42, 'Grindles': 43, 'HAD': 44, 'Had': 45, 'Hang': 46, 'Has': 47, 'He': 48, 'Her': 49, 'Hermia': 50, 'His': 51, 'How': 52, 'I': 53, 'If': 54, 'In': 55, 'It': 56, 'Jack': 57, 'Jove': 58, 'Just': 59, 'Lord': 60, 'Made': 61, 'Miss': 62, 'Money': 63, 'Monte': 64, 'Moon-dancers': 65, 'Mr': 66, 'Mrs': 67, 'My': 68, 'Never': 69, 'No': 70, 'Now': 71, 'Nutley': 72, 'Of': 73, 'Oh': 74, 'On': 75, 'Once': 76, 'Only': 77, 'Or': 78, 'Perhaps': 79, 'Poor': 80, 'Professional': 81, 'Renaissance': 82, 'Ri

In [17]:
#Eg:
vocab["Jack"]    #we r doing str_to_int

57

In [18]:
# reverse Mapping:
int_to_str = {i:s for s,i in vocab.items()}
int_to_str[57]

# These 2 cells show what's happening in the encode fun() below

'Jack'

In [19]:
class simpleTokenizerV1:
    def __init__(self,vocab):                                         #init is the construstor(automatically called at thr time of object creation)
        self.str_to_int = vocab                                       #to store string to integer mapping. Eg: 'Mr' : 66
        self.int_to_str = {i:s for s,i in vocab.items()}              #reverse the mapping made above

    def encode(self,text):
        preprocessed = re.split(r'([,.:;?_"!()\']|--|\s)',text)                   #using regular expression
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()                # removing/ stripe off all the white spaces
        ]
        ids = [self.str_to_int[s] for s in preprocessed]                         #creating the token IDs. We r using the vocab from (line 4) for each token of string in preprocessed is converted into token IDs[Eg: in the above 2 cells] 
        return ids
        
    def decode(self,ids):                                            # like reverse mapping of what done in encode
        text = " ".join([self.int_to_str[i] for i in ids])            # here we r going to recreate the text from going from integer to txt in vocab for each token IDs
        text = re.sub(r'\s+([,.?!"()])', r'\1', text)                 # Replace spaces before the specified punctuations
        return text

In [20]:
tokenizer = simpleTokenizerV1(vocab)

In [21]:
txt = """"It's the last he painted, you know,"
        Mrs. Gisburn said with pardonable pride"""

In [22]:
ids = tokenizer.encode(txt)
print(ids)                         # the below is the representation of the txt in above cell

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793]


In [23]:
tokenizer.decode(ids)

'" It \' s the last he painted, you know," Mrs. Gisburn said with pardonable pride'

# Adding special context tokens

In [25]:
text = "Hello, do you like tea. This is a --test"
tokenizer.encode(text)                                    #Here it'll give an error cause the word "Hello is not in the vocab"

KeyError: 'Hello'

In [ ]:
all_tokens = sorted(list(set(preProcessed)))       
all_tokens.extend(["<|endoftext|>","<|unk|>"])      #We are adding two more words for to handle end of text and unknown characters

vocab = {token : integer for integer,token in enumerate(all_tokens)}

In [ ]:
len(vocab)         #before it was 1130 since we added for end of text and unknow characters

In [ ]:
for i,item in enumerate(list(vocab.items())[-5:]):
    print(item)                                               #can check the last 2 newly added special tokens

In [ ]:
class simpleTokenizerV2:                                                      #new Version simpleTokenizerV2
    def __init__(self,vocab):                                         
        self.str_to_int = vocab                                       
        self.int_to_str = {i:s for s,i in vocab.items()}              

    def encode(self,text):
        preprocessed = re.split(r'([,.:;?_"!()\']|--|\s)',text)                   
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()                
        ]
        preprocessed = [                                            
            item if item in self.str_to_int                                       #Added new rule to handle unknow words
            else "<|unk|>" for item in preprocessed
        ]
        ids = [self.str_to_int[s] for s in preprocessed]    
        return ids
        
    def decode(self,ids):                                             
        text = " ".join([self.int_to_str[i] for i in ids])            
        text = re.sub(r'\s+([,.?!"()])', r'\1', text)                 
        return text

In [ ]:
tokenizer = simpleTokenizerV2(vocab)

In [ ]:
text = "Hello, do you like tea. This is a --test"
tokenizer.encode(text)                                     #Now the tokenizer is handling the unknown word that it didn't handle in V1

In [ ]:
tokenizer.decode(tokenizer.encode(text))                   #As the modified v2 tokenizer hankels the unknown words ,
                                                           #you can see when we decode it  the unknown words are marked as<|unk| 

# Byte pair encoding

In [ ]:
uv pip install tiktoken

In [1]:
import tiktoken                           #tiktoken lib helps us to use the gpt-2 tokenizer (using gpt-2 cause to make out llm mode compact here)
# if haven't installed the tiktoken ---> !pip install tiktoken

In [5]:
tiktoken.__version__

'0.14.0'

In [9]:
tokenizer = tiktoken.get_encoding("gpt2")          #it will have evething like all vocab, implemenation of this tokenizer

In [11]:
tokenizer.encode("hello world")

[31373, 995]

In [ ]:
tokenizer.decode(tokenizer.encode("hello world"))

In [9]:
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
    "of someunknownPlace"
)
tokenizer.encode(text)                 #It'll raise an error cause <|endoftext|> is used to concat two documents; since we use it at the midde ot the sentence it'll show an error

ValueError: Encountered text corresponding to disallowed special token '<|endoftext|>'.
If you want this text to be encoded as a special token, pass it to `allowed_special`, e.g. `allowed_special={'<|endoftext|>', ...}`.
If you want this text to be encoded as normal text, disable the check for this token by passing `disallowed_special=(enc.special_tokens_set - {'<|endoftext|>'})`.
To disable this check for all special tokens, pass `disallowed_special=()`.


In [11]:
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
    "of someunknownPlace"
)
tokenizer.encode(text,allowed_special= {"<|endoftext|>"})                 

[15496,
 11,
 466,
 345,
 588,
 8887,
 30,
 220,
 50256,
 554,
 262,
 4252,
 18250,
 8812,
 2114,
 1659,
 617,
 34680,
 27271]

In [13]:
"""Example mention in the explanation of the byte pair encoding, as will break down arbitrary text into subwords
    We'll use the same exapmle here mention above (explanation)"""


unknown = "unknownwordsasbiteonhfenfb"
tokenizer.encode(unknown)

[34680, 10879, 292, 37018, 261, 71, 41037, 21855]

In [15]:
tokenizer.decode([34680, 10879, 292, 37018, 261, 71, 41037, 21855])

'unknownwordsasbiteonhfenfb'

In [17]:
"""
    if you want to build this byte pair encoding from scratch, go to: https://github.com/rasbt/LLMs-from-scratch/blob/main/ch02/05_bpe-from-scratch/bpe-from-scratch-simple.ipynb
    """

'\n    if you want to build this byte pair encoding from scratch, go to: https://github.com/rasbt/LLMs-from-scratch/blob/main/ch02/05_bpe-from-scratch/bpe-from-scratch-simple.ipynb\n    '

# Data Sampling with sliding window

In [3]:
with open("the-verdict.txt","r",encoding="utf-8") as f:
    raw_text = f.read()
enc_text = tokenizer.encode(raw_text)                #Encoding to token id using tiktiken
print(len(enc_text))

NameError: name 'tokenizer' is not defined

In [24]:
enc_sample = enc_text[50:]          #truncating the 1stt 50 tokens

In [30]:
len(enc_sample)

5095

In [38]:
context_size = 4

x = enc_sample[:context_size]                        #inputs
y = enc_sample[1:context_size + 1]                   #targets

print(f" x : {x}")
print(f" y :      {y}")

 x : [290, 4920, 2241, 287]
 y :      [4920, 2241, 287, 257]


In [44]:
# FOR A CLEARER VISUALIZATION, HERE IS A DETAIL EG:
for i in range(1,context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(context, "-----> " ,desired)

[290] ----->  4920
[290, 4920] ----->  2241
[290, 4920, 2241] ----->  287
[290, 4920, 2241, 287] ----->  257


In [46]:
# Decoding th above
for i in range(1,context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(tokenizer.decode(context), "-----> " ,tokenizer.decode([desired]))

 and ----->   established
 and established ----->   himself
 and established himself ----->   in
 and established himself in ----->   a


## To make this efficient we are using pytorch 

In [5]:
import torch         #Pytorch is imported as torch (default) lol i cant do anything 

In [7]:
torch.__version__     # to install some specific version pip install torch == 2.3.1 (or anything)

'2.3.1+cpu'

In [9]:
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenizing the entire text
        token_ids = tokenizer.encode(
            txt, allowed_special={"<|endoftext|>"}
        )

        # Sliding window to create overlapping sequences
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1:i + max_length + 1]

            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [11]:
def create_dataloader_v1(
    txt,
    batch_size=4,
    max_length=256,
    stride=128,
    shuffle=True,
    drop_last=True,
    num_workers=0
):
    # Initialize tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

In [13]:
with open("the-verdict.txt","r",encoding="utf-8") as f:
    raw_text = f.read()

In [15]:
dataloader = create_dataloader_v1(
    raw_text,batch_size = 1, max_length = 4 , stride = 4 , shuffle = False           #stride means how many places we are moving the sliding window
)

data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


In [27]:
second_batch = next(data_iter)
print(second_batch)

[tensor([[1807, 3619,  402,  271]]), tensor([[ 3619,   402,   271, 10899]])]


In [29]:
# training with batch size of 8
dataloader = create_dataloader_v1(
    raw_text,batch_size = 8, max_length = 4 , stride = 4 , shuffle = False           #stride means how many places we are moving the sliding window
)
data_iter = iter(dataloader)
inp , tar = next(data_iter)
print("Inputs:\n",inp)
print("\nTargets:\n",tar)

Inputs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Targets:
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


# Creating token embeddings

In [32]:
# An  examle to understand:

inp_ids = torch.tensor([2,3,5,1])

In [34]:
vocab_size = 6
output_dim = 3

torch.manual_seed(123) 
embedding_layer = torch.nn.Embedding(vocab_size,output_dim)                #torch.nn.Embedding <--- its a neural network layer with random weights and random weights are differnet when you iniitate each time. SO in order to get the same out as me , using torch.manual_seed(123)

In [36]:
print(embedding_layer.weight)

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)


In [40]:
embedding_layer(torch.tensor([3]))                  #have look ok the op i.e: [-0.4015,  0.9666, -1.1481] we are getting the weights from the above vectors  which corresponds to the input index . NOTE : INDEXSS

tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)

In [42]:
embedding_layer(torch.tensor([2]))                  # have a looke at the index 2 in the above matrix

tensor([[ 1.2753, -0.2010, -0.1606]], grad_fn=<EmbeddingBackward0>)

In [45]:
embedding_layer(inp_ids)

tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)

In [47]:
inp_ids

tensor([2, 3, 5, 1])

# Encoding Word Position

In [17]:
vocab_size = 50257                        # actual vocab size (tokens/words)
output_dim = 256                             # means 50257 rows and 256 cols
 
token_embedding_layer = torch.nn.Embedding(vocab_size,output_dim)

In [19]:
max_length = 4
dataloader = create_dataloader_v1(
    raw_text ,batch_size=8, max_length = max_length,
    stride= max_length , shuffle= False
)

data_iter = iter(dataloader)
inputs , targets = next(data_iter)

In [21]:
print("Token IDs: \n", inputs)
print("\n Input Shape: \n", inputs.shape)


Token IDs: 
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

 Input Shape: 
 torch.Size([8, 4])


In [23]:
token_embeddings = token_embedding_layer(inputs)
token_embeddings.shape                                  # gives a 3d tensors

torch.Size([8, 4, 256])

In [25]:
# just visualize 1 embedding
token_embeddings[0,0]                    # <---corresponds to 1st batch 1st vector    ,<--- o\p willl be the 256 dim vector

tensor([-1.4989e-01, -2.7420e-01,  1.3371e-02, -2.0924e+00,  1.5741e+00,
        -2.5487e-01,  4.4053e-02,  6.4302e-01,  1.7339e+00,  9.6769e-01,
        -1.2420e-01, -3.3072e-01, -1.5291e+00, -4.3391e-01,  8.5540e-01,
         2.5947e+00,  4.6361e-01,  1.0681e+00,  1.1502e+00, -2.8576e-02,
         1.2612e+00, -5.7057e-02, -9.6029e-01,  9.8725e-01,  2.0846e-01,
        -1.6354e+00,  5.9547e-01,  1.5157e-01,  4.8471e-01,  3.8505e-01,
         2.5081e+00, -9.5503e-01, -2.4402e-01,  7.0047e-01, -8.7884e-01,
         1.2841e+00, -7.3129e-01,  1.0836e+00, -6.7211e-01, -1.9267e+00,
         2.8902e-01, -7.3197e-01, -5.8773e-02, -1.0281e+00, -5.3307e-01,
         2.2748e-01,  6.0424e-01, -2.9112e-01,  2.4342e-01,  5.5101e-01,
         9.6572e-02, -1.5344e+00, -1.8637e+00,  1.0706e+00,  4.2335e-01,
         5.9249e-01,  2.6210e-01,  6.7278e-01, -2.0045e+00, -4.4183e-01,
         4.2311e-01, -6.4961e-01,  7.6040e-01, -5.4380e-01, -8.2578e-01,
         1.5114e+00, -9.1181e-01, -5.4836e-01,  6.0

In [27]:
### ADDING POSITIONAL LAYER
context_len = max_length
pos_embedding_layer = torch.nn.Embedding(context_len,output_dim)

In [33]:
torch.arange(max_length)

tensor([0, 1, 2, 3])

In [35]:
pos_embedding_layer.weight

Parameter containing:
tensor([[-0.9749, -0.2054, -0.3096,  ..., -1.9706,  1.4663,  0.4756],
        [-1.6926, -0.2720,  1.7727,  ..., -0.6749,  0.4889,  1.0023],
        [ 1.2260, -0.3178,  0.6907,  ..., -0.2329, -0.0353, -0.5598],
        [-1.9561,  0.3576,  1.4572,  ...,  0.2054,  0.4151, -0.0896]],
       requires_grad=True)

In [37]:
pos_embedding_layer(torch.arange(max_length))

tensor([[-0.9749, -0.2054, -0.3096,  ..., -1.9706,  1.4663,  0.4756],
        [-1.6926, -0.2720,  1.7727,  ..., -0.6749,  0.4889,  1.0023],
        [ 1.2260, -0.3178,  0.6907,  ..., -0.2329, -0.0353, -0.5598],
        [-1.9561,  0.3576,  1.4572,  ...,  0.2054,  0.4151, -0.0896]],
       grad_fn=<EmbeddingBackward0>)

In [39]:
pos_embedding = pos_embedding_layer(torch.arange(max_length))
print(pos_embedding.shape)

torch.Size([4, 256])


In [41]:
token_embeddings.shape

torch.Size([8, 4, 256])

In [43]:
pos_embedding.shape

torch.Size([4, 256])

In [45]:
#EG for size diff
token_embeddings[0] + pos_embedding

tensor([[-1.1248, -0.4796, -0.2962,  ..., -2.5633,  1.6329,  0.7632],
        [-0.9247,  0.7207,  0.6540,  ..., -1.0889,  2.0070,  0.7588],
        [ 1.4756, -1.2423, -0.0104,  ..., -1.4205, -0.1044, -1.4900],
        [-1.7958,  2.3758,  1.2705,  ...,  0.9840, -0.3216,  1.1443]],
       grad_fn=<AddBackward0>)

In [49]:
token_embeddings + pos_embedding        #why the diff in token_embeddings.shape and  pos_embedding.shape is the pytorch is  some wahts 
                                        #smart in handling the token embeddings; as if the sizes are the same, it won't duplicate and add again
                                        #for visualization look at the o/p fir this cell and the above wit 1st vector in token_embeddings

tensor([[[-1.1248, -0.4796, -0.2962,  ..., -2.5633,  1.6329,  0.7632],
         [-0.9247,  0.7207,  0.6540,  ..., -1.0889,  2.0070,  0.7588],
         [ 1.4756, -1.2423, -0.0104,  ..., -1.4205, -0.1044, -1.4900],
         [-1.7958,  2.3758,  1.2705,  ...,  0.9840, -0.3216,  1.1443]],

        [[-2.1498,  0.8677, -0.0647,  ..., -1.7028,  2.1725, -0.1084],
         [-1.5694, -0.3983,  0.4877,  ..., -1.0461,  1.8861,  1.2444],
         [ 1.2764, -1.3111,  1.0573,  ...,  1.2736,  0.4034,  1.3478],
         [-1.2122,  1.8852,  1.4286,  ..., -0.0552, -0.1630, -0.9977]],

        [[-2.1149,  0.1710, -0.3450,  ...,  0.6092,  0.6661, -0.5135],
         [-1.3596, -2.2572,  1.9185,  ..., -2.6049,  0.3965,  3.6643],
         [ 1.5854, -1.8812,  1.4272,  ..., -0.9747,  1.1897, -1.5833],
         [-1.7750, -0.5332, -0.3470,  ...,  0.2259, -0.5755,  1.0314]],

        ...,

        [[-0.7012,  0.6097,  0.3476,  ..., -1.5998,  1.9754,  1.5911],
         [-1.3145,  0.5478,  1.1252,  ..., -0.9974,  1.38

In [51]:
input_embedding  = token_embeddings + pos_embedding  
print(input_embedding.shape)

torch.Size([8, 4, 256])
